# Functional Determinism: Retriever vs Pub/Sub

**Retriever** uses event-time semantics (logical clock → fixed execution order → identical traces).  
**Pub/Sub** uses arrival-time semantics (message jitter → corrupted traces → wrong gradients).

| | Retriever | Pub/Sub |
|--|--|--|
| Clock | logical (fixed order) | arrival-time (jittered) |
| Trace | identical every run | differs each run |
| ∂L/∂θ | correct | corrupted |

**System**: 1D bouncing ball — optimize `θ = v₀` (initial velocity) to minimize `L = (x_T - x*)²`

In [ ]:
import os, sys
import numpy as np
import torch
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from dataclasses import dataclass
from typing import List, Tuple, Optional
%matplotlib inline

# Add project root so we can import retriever
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from retriever.flow import Flow, Pipeline, Rate, Trigger, io

## Physics & Gradient Computation

In [ ]:
@dataclass
class PhysicsConfig:
    """Shared config for bouncing ball experiments."""
    g: float = 9.81       # gravity
    e: float = 0.8        # coefficient of restitution
    dt: float = 0.01      # time step
    T: int = 100           # horizon (steps)
    x_init: float = 1.0   # initial height
    x_target: float = 0.5 # target height
    theta: float = 3.0    # initial velocity (optimized parameter)


class BouncingBall:
    """Bouncing ball hybrid system with differentiable gradient computation."""

    def __init__(self, cfg: PhysicsConfig):
        self.cfg = cfg

    def simulate(self, theta_val: float) -> Tuple[np.ndarray, np.ndarray, List[bool]]:
        """Forward simulation (numpy). Returns (times, positions, contacts)."""
        c = self.cfg
        x, v = c.x_init, theta_val
        times, xs, contacts = [0.0], [x], []
        for t in range(c.T):
            v_pred = v - c.g * c.dt
            x_pred = x + v_pred * c.dt
            contact = x_pred < 0.0
            x, v = (0.0, -c.e * v_pred) if contact else (x_pred, v_pred)
            times.append((t + 1) * c.dt)
            xs.append(x)
            contacts.append(contact)
        return np.array(times), np.array(xs), contacts

    def gradient(self, contacts: List[bool]) -> Tuple[float, float]:
        """Compute ∂L/∂θ via PyTorch autograd given a contact sequence."""
        c = self.cfg
        theta = torch.tensor(c.theta, dtype=torch.float64, requires_grad=True)
        x = torch.tensor(c.x_init, dtype=torch.float64)
        v = theta
        for contact in contacts:
            v_pred = v - c.g * c.dt
            x_pred = x + v_pred * c.dt
            ct = torch.tensor(contact)
            x = torch.where(ct, x_pred * 0, x_pred)  # keep graph connected
            v = torch.where(ct, -c.e * v_pred, v_pred)
        loss = (x - c.x_target) ** 2
        loss.backward()
        return theta.grad.item(), loss.item()

    def analytical_gradient(self, eps=1e-6) -> float:
        """Finite-difference reference gradient."""
        c = self.cfg
        def loss(th):
            x, v = c.x_init, th
            for _ in range(c.T):
                v_pred = v - c.g * c.dt
                x_pred = x + v_pred * c.dt
                x, v = (0.0, -c.e * v_pred) if x_pred < 0 else (x_pred, v_pred)
            return (x - c.x_target) ** 2
        return (loss(c.theta + eps) - loss(c.theta)) / eps

print("✓ BouncingBall loaded")

## Retriever Flows (actual `Pipeline.run()`)

These are real Retriever `Flow` classes wired with `Rate`/`Trigger` and executed via `Pipeline.run()`.

In [ ]:
# ── Retriever I/O types ───────────────────────────────────────────────────────

@io
class ClockTick:
    tick: Optional[int] = None
    dt: Optional[float] = None

@io
class BallState:
    tick: Optional[int] = None
    x: Optional[float] = None
    v: Optional[float] = None
    contact: Optional[bool] = None

@io
class GradientResult:
    gradient: Optional[float] = None
    loss: Optional[float] = None

# ── Retriever Flows ───────────────────────────────────────────────────────────

class TickClockFlow(Flow[None, ClockTick]):
    def __init__(self, dt: float, max_ticks: int):
        super().__init__()
        self.dt, self.max_ticks = float(dt), int(max_ticks)

    def init_config(self): return {"dt": self.dt, "max_ticks": self.max_ticks}
    def init(self): self.tick = 0

    def step(self, _: None) -> ClockTick:
        if self.tick >= self.max_ticks:
            return ClockTick()
        t = self.tick; self.tick += 1
        return ClockTick(tick=t, dt=self.dt)


class BouncingBallFlow(Flow[ClockTick, BallState]):
    def __init__(self, g: float, e: float, x_init: float, theta: float):
        super().__init__()
        self.g, self.e = float(g), float(e)
        self.x_init, self.theta = float(x_init), float(theta)

    def init_config(self): return {"g": self.g, "e": self.e, "x_init": self.x_init, "theta": self.theta}

    def init(self):
        self.x, self.v = self.x_init, self.theta

    def step(self, inp: ClockTick) -> BallState:
        if inp.tick is None: return BallState()
        v_pred = self.v - self.g * float(inp.dt)
        x_pred = self.x + v_pred * float(inp.dt)
        contact = x_pred < 0.0
        self.x, self.v = (0.0, -self.e * v_pred) if contact else (x_pred, v_pred)
        return BallState(tick=inp.tick, x=self.x, v=self.v, contact=contact)


class TraceGradientFlow(Flow[BallState, GradientResult]):
    def __init__(self, g: float, e: float, x_init: float, x_target: float, theta: float, max_ticks: int, dt: float):
        super().__init__()
        self.g, self.e, self.dt = float(g), float(e), float(dt)
        self.x_init, self.x_target, self.theta = float(x_init), float(x_target), float(theta)
        self.max_ticks = int(max_ticks)

    def init_config(self):
        return dict(g=self.g, e=self.e, x_init=self.x_init, x_target=self.x_target,
                    theta=self.theta, max_ticks=self.max_ticks, dt=self.dt)

    def init(self):
        self.contacts: List[bool] = []

    def step(self, inp: BallState) -> GradientResult:
        if inp.tick is None: return GradientResult()
        self.contacts.append(inp.contact or False)
        if inp.tick >= self.max_ticks - 1:
            return self._compute()
        return GradientResult()

    def _compute(self) -> GradientResult:
        theta = torch.tensor(self.theta, dtype=torch.float64, requires_grad=True)
        x = torch.tensor(self.x_init, dtype=torch.float64)
        v = theta
        for contact in self.contacts:
            v_pred = v - self.g * self.dt
            x_pred = x + v_pred * self.dt
            c = torch.tensor(contact)
            x = torch.where(c, x_pred * 0, x_pred)
            v = torch.where(c, -self.e * v_pred, v_pred)
        loss = (x - self.x_target) ** 2
        loss.backward()
        return GradientResult(gradient=theta.grad.item(), loss=loss.item())


class ResultSinkFlow(Flow[GradientResult, None]):
    def init(self): self.result = None
    def step(self, inp: GradientResult) -> None:
        if inp.gradient is not None:
            self.result = inp

# ── Pipeline builder ──────────────────────────────────────────────────────────

def run_retriever_pipeline(cfg: PhysicsConfig) -> Tuple[float, float]:
    """Run actual Retriever Pipeline.run() and return (gradient, loss)."""
    clock = TickClockFlow(dt=cfg.dt, max_ticks=cfg.T)
    ball  = BouncingBallFlow(g=cfg.g, e=cfg.e, x_init=cfg.x_init, theta=cfg.theta)
    trace = TraceGradientFlow(g=cfg.g, e=cfg.e, x_init=cfg.x_init, x_target=cfg.x_target,
                               theta=cfg.theta, max_ticks=cfg.T, dt=cfg.dt)
    sink  = ResultSinkFlow()

    pipe = Pipeline("bouncing_ball")
    with pipe:
        h_clock = clock @ Rate(hz=int(1/cfg.dt))
        h_ball  = ball  @ Trigger("tick")
        h_trace = trace @ Trigger("x")
        h_sink  = sink  @ Trigger("gradient")
        h_clock >> h_ball >> h_trace >> h_sink

    pipe.run(backend="in-process", duration=cfg.T * cfg.dt + 0.5, blocking=True)

    r = sink.result
    return (r.gradient, r.loss) if r else (float('nan'), float('nan'))

print("✓ Retriever Flows loaded")

## Pub/Sub Executor (emulated arrival-time semantics)

In [ ]:
class PubSubExecutor:
    """Emulates arrival-time semantics: subscribers sometimes read stale state."""

    def __init__(self, ball: BouncingBall, jitter_prob: float = 0.2, seed: int = 42):
        self.ball = ball
        self.jitter_prob = jitter_prob
        self.rng = np.random.default_rng(seed)

    def run(self) -> Tuple[np.ndarray, np.ndarray, float, float]:
        """Run one trial. Returns (times, positions, gradient, loss)."""
        c = self.ball.cfg
        x, v = c.x_init, c.theta
        buf_x, buf_v = x, v
        times, xs, contacts = [0.0], [x], []

        for t in range(c.T):
            if self.rng.random() < self.jitter_prob:
                rx, rv = buf_x, buf_v   # stale read
            else:
                rx, rv = x, v
                buf_x, buf_v = x, v

            v_pred = rv - c.g * c.dt
            x_pred = rx + v_pred * c.dt
            contact = x_pred < 0.0
            x, v = (0.0, -c.e * v_pred) if contact else (x_pred, v_pred)
            times.append((t + 1) * c.dt)
            xs.append(x)
            contacts.append(contact)

        grad, loss = self.ball.gradient(contacts)
        return np.array(times), np.array(xs), grad, loss

print("✓ PubSubExecutor loaded")

## 1 · Single run — trajectory comparison

**Left**: Retriever (actual `Pipeline.run()`). **Right**: Pub/Sub (jittered).

In [ ]:
def plot_single(theta=3.0, jitter_prob=0.2, T=100, x_target=0.5, seed=0):
    cfg = PhysicsConfig(theta=theta, T=T, x_target=x_target)
    ball = BouncingBall(cfg)
    true_grad = ball.analytical_gradient()

    # Retriever: actual Pipeline.run()
    r_grad, r_loss = run_retriever_pipeline(cfg)
    r_times, r_xs, _ = ball.simulate(theta)

    # Pub/Sub: jittered
    ps = PubSubExecutor(ball, jitter_prob=jitter_prob, seed=seed)
    p_times, p_xs, p_grad, _ = ps.run()

    fig, axes = plt.subplots(1, 2, figsize=(7, 2.8), sharey=True)
    for ax, times, xs, grad, title, color in [
        (axes[0], r_times, r_xs, r_grad, 'Retriever (Pipeline.run)', '#2563eb'),
        (axes[1], p_times, p_xs, p_grad, 'Pub/Sub (jittered)',       '#f97316'),
    ]:
        ax.plot(times, xs, color=color, linewidth=1.3)
        ax.axhline(x_target, color='#dc2626', linewidth=1.2, linestyle='--', alpha=0.8)
        ax.axhline(0, color='#1f2937', linewidth=2.5)
        err = abs(grad - true_grad)
        ax.set_title(f'{title}\n∂L/∂θ = {grad:.4f}   err = {err:.2e}', fontsize=9)
        ax.set_xlabel('time (s)', fontsize=8); ax.tick_params(labelsize=7)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    axes[0].set_ylabel('height', fontsize=8)
    axes[0].text(0.02, 0.93, f'true ∂L/∂θ = {true_grad:.4f}', transform=axes[0].transAxes,
                 fontsize=7, color='#dc2626')
    plt.tight_layout(); plt.show()

display(widgets.interactive(plot_single,
    theta      = widgets.FloatSlider(value=3.0, min=1.0, max=6.0, step=0.1,  description='θ (v₀)'),
    jitter_prob= widgets.FloatSlider(value=0.2, min=0.0, max=0.8, step=0.05, description='jitter'),
    T          = widgets.IntSlider(  value=100, min=50,  max=200, step=10,   description='T steps'),
    x_target   = widgets.FloatSlider(value=0.5, min=0.1, max=1.5, step=0.1,  description='x* target'),
    seed       = widgets.IntSlider(  value=0,   min=0,   max=200, step=1,    description='PS seed'),
))

## 2 · Multi-run — gradient distribution

In [ ]:
def plot_gradient_dist(theta=3.0, K=50, jitter_prob=0.2, T=100, x_target=0.5, seed=42):
    cfg = PhysicsConfig(theta=theta, T=T, x_target=x_target)
    ball = BouncingBall(cfg)
    true_grad = ball.analytical_gradient()
    r_grad, _ = run_retriever_pipeline(cfg)

    rng = np.random.default_rng(seed)
    ps_grads = []
    for _ in range(K):
        ps = PubSubExecutor(ball, jitter_prob=jitter_prob, seed=rng.integers(1_000_000))
        _, _, g, _ = ps.run()
        ps_grads.append(g)

    fig, ax = plt.subplots(figsize=(5, 2.8))
    n_unique = len(set(f'{g:.8f}' for g in ps_grads))
    ax.hist(ps_grads, bins=min(n_unique, 40), color='#f97316', alpha=0.75, edgecolor='white',
            label=f'Pub/Sub  μ={np.mean(ps_grads):.3f}  σ={np.std(ps_grads):.3f}')
    ax.axvline(true_grad, color='#dc2626', linewidth=2, linestyle='--', label=f'True: {true_grad:.4f}')
    ax.axvline(r_grad,    color='#2563eb', linewidth=2.5,               label=f'Retriever: {r_grad:.4f}')
    ax.set_xlabel(r'$\partial L / \partial \theta$', fontsize=9)
    ax.set_ylabel('count', fontsize=9)
    ax.set_title(f'Gradient distribution  (K={K} pub/sub runs)', fontsize=9)
    ax.legend(fontsize=7); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.tight_layout(); plt.show()

display(widgets.interactive(plot_gradient_dist,
    theta      = widgets.FloatSlider(value=3.0, min=1.0, max=6.0, step=0.1,  description='θ (v₀)'),
    K          = widgets.IntSlider(  value=50,  min=10,  max=200, step=10,   description='K runs'),
    jitter_prob= widgets.FloatSlider(value=0.2, min=0.0, max=0.8, step=0.05, description='jitter'),
    T          = widgets.IntSlider(  value=100, min=50,  max=200, step=10,   description='T steps'),
    x_target   = widgets.FloatSlider(value=0.5, min=0.1, max=1.5, step=0.1,  description='x* target'),
    seed       = widgets.IntSlider(  value=42,  min=0,   max=999, step=1,    description='seed'),
))

## 3 · Gradient error vs jitter probability

In [ ]:
def plot_jitter_sweep(theta=3.0, K=30, T=100, x_target=0.5, seed=42):
    cfg = PhysicsConfig(theta=theta, T=T, x_target=x_target)
    ball = BouncingBall(cfg)
    true_grad = ball.analytical_gradient()
    r_grad, _ = run_retriever_pipeline(cfg)

    jitter_probs = np.linspace(0.0, 0.7, 15)
    rng = np.random.default_rng(seed)
    means, stds = [], []
    for jp in jitter_probs:
        grads = [PubSubExecutor(ball, jitter_prob=jp, seed=rng.integers(1_000_000)).run()[2]
                 for _ in range(K)]
        means.append(np.mean(grads)); stds.append(np.std(grads))
    means, stds = np.array(means), np.array(stds)

    fig, ax = plt.subplots(figsize=(5, 2.8))
    ax.fill_between(jitter_probs, means - stds, means + stds, color='#f97316', alpha=0.2)
    ax.plot(jitter_probs, means, color='#f97316', linewidth=1.8, label='Pub/Sub mean ± 1σ')
    ax.axhline(true_grad, color='#dc2626', linewidth=1.5, linestyle='--', label=f'True: {true_grad:.4f}')
    ax.axhline(r_grad,    color='#2563eb', linewidth=1.5,                label=f'Retriever: {r_grad:.4f}')
    ax.set_xlabel('jitter probability', fontsize=9)
    ax.set_ylabel(r'$\partial L / \partial \theta$', fontsize=9)
    ax.set_title('Gradient error vs jitter', fontsize=9)
    ax.legend(fontsize=7); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.tight_layout(); plt.show()

display(widgets.interactive(plot_jitter_sweep,
    theta   = widgets.FloatSlider(value=3.0, min=1.0, max=6.0, step=0.1, description='θ (v₀)'),
    K       = widgets.IntSlider(  value=30,  min=10,  max=100, step=5,   description='K runs'),
    T       = widgets.IntSlider(  value=100, min=50,  max=200, step=10,  description='T steps'),
    x_target= widgets.FloatSlider(value=0.5, min=0.1, max=1.5, step=0.1, description='x* target'),
    seed    = widgets.IntSlider(  value=42,  min=0,   max=999, step=1,   description='seed'),
))